# 03 — WGI Governance Compilation

This notebook comes after:

`02_GDP_Recovery_Dynamic_Baseline.ipynb`

Purpose:

Process the newly downloaded World Bank WGI CSV/DataBank package into a clean country-year governance dataset.

This version is robust to two WGI CSV package formats:

1. packages containing both WGI estimates and percentile-rank / 0–100 scores;
2. packages containing only WGI estimates.

The current WGI package schema uses codes such as `GOV_WGI_CC.SC`, `GOV_WGI_CC.EST`, `GOV_WGI_CC.SE`, and `GOV_WGI_CC.SR`.

This notebook does:

- inspect the extracted WGI CSV package;
- identify the main WGI data file automatically;
- extract the six WGI governance dimensions;
- extract available WGI measures;
- reshape WGI into long and wide country-year formats;
- run coverage diagnostics;
- compute six-dimension correlation matrices;
- compute Euclidean and Mahalanobis distance-to-ideal governance composites;
- export cleaned WGI outputs and diagnostics.

This notebook does **not**:

- use the old six-sheet Excel workbook;
- merge WGI into the master dataset;
- choose the final country sample;
- create POSet variables;
- use WGI as a final scalar resilience index.

Important:

The six WGI dimensions are official WGI variables.  
The Mahalanobis governance score is a **derived project composite**, not an official WGI variable.

If the CSV package does not contain 0–100 governance score rows, the notebook uses WGI estimates directly for the governance composite.

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import json
import shutil
import zipfile
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

RAW_FILES_DIR = PROJECT_ROOT / "Data" / "Raw" / "Raw_Files"

WGI_ZIP_FILE = RAW_FILES_DIR / "WGI_CSV.zip"
WGI_RAW_DIR = RAW_FILES_DIR / "WGI_2025_Raw"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "WGI_Governance"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "03_WGI_Governance_Compilation"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 1996
PROJECT_START_YEAR = 2000
PROJECT_END_YEAR = 2024

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("WGI zip file:", WGI_ZIP_FILE.resolve())
print("WGI raw folder:", WGI_RAW_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())

Run ID: 20260622_032632
WGI zip file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Raw\Raw_Files\WGI_CSV.zip
WGI raw folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Raw\Raw_Files\WGI_2025_Raw
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\WGI_Governance
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\03_WGI_Governance_Compilation


In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

WGI_DIMENSIONS = {
    "VA": {
        "slug": "va",
        "name": "Voice and Accountability",
        "name_patterns": [
            "voice and accountability",
            "voice accountability",
        ],
    },
    "PV": {
        "slug": "pv",
        "name": "Political Stability and Absence of Violence/Terrorism",
        "name_patterns": [
            "political stability",
            "absence of violence",
            "terrorism",
        ],
    },
    "GE": {
        "slug": "ge",
        "name": "Government Effectiveness",
        "name_patterns": [
            "government effectiveness",
        ],
    },
    "RQ": {
        "slug": "rq",
        "name": "Regulatory Quality",
        "name_patterns": [
            "regulatory quality",
        ],
    },
    "RL": {
        "slug": "rl",
        "name": "Rule of Law",
        "name_patterns": [
            "rule of law",
        ],
    },
    "CC": {
        "slug": "cc",
        "name": "Control of Corruption",
        "name_patterns": [
            "control of corruption",
            "corruption",
        ],
    },
}

# Current WGI CSV package series codes follow this pattern:
#
# GOV_WGI_CC.EST       Governance estimate, approx. -2.5 to +2.5
# GOV_WGI_CC.SC        Governance score, 0-100
# GOV_WGI_CC.SC_LB     Lower bound of 90% CI for governance score
# GOV_WGI_CC.SC_UB     Upper bound of 90% CI for governance score
# GOV_WGI_CC.SE        Standard error of governance estimate
# GOV_WGI_CC.SR        Number of sources
#
# The rules below also include a few older/common aliases to keep the notebook robust.

WGI_MEASURE_RULES = [
    {
        "code_suffixes": [".SC_LB", ".PER.RNK.LOWER", ".PR.LOWER", ".PERCENTILE.RANK.LOWER"],
        "name_patterns": [
            "lower bound of the 90% confidence interval for the governance score",
            "lower bound",
            "score lower",
            "percentile rank lower",
            "percentile rank, lower",
            "lower percentile rank",
        ],
        "measure": "score_lower",
        "wide_suffix": "score_lower",
    },
    {
        "code_suffixes": [".SC_UB", ".PER.RNK.UPPER", ".PR.UPPER", ".PERCENTILE.RANK.UPPER"],
        "name_patterns": [
            "upper bound of the 90% confidence interval for the governance score",
            "upper bound",
            "score upper",
            "percentile rank upper",
            "percentile rank, upper",
            "upper percentile rank",
        ],
        "measure": "score_upper",
        "wide_suffix": "score_upper",
    },
    {
        "code_suffixes": [".SC", ".PER.RNK", ".PR", ".PERCENTILE.RANK"],
        "name_patterns": [
            "governance score",
            "score (0 100)",
            "score 0 100",
            "0 100",
            "percentile rank",
        ],
        "measure": "score",
        "wide_suffix": "score",
    },
    {
        "code_suffixes": [".SE", ".STD.ERR", ".STANDARD.ERROR"],
        "name_patterns": ["standard error"],
        "measure": "standard_error",
        "wide_suffix": "std_error",
    },
    {
        "code_suffixes": [".SR", ".NO.SRC", ".NOSRC", ".NUMBER.SOURCES"],
        "name_patterns": ["number of sources", "no. of sources", "no of sources"],
        "measure": "number_of_sources",
        "wide_suffix": "num_sources",
    },
    {
        "code_suffixes": [".EST", ".ESTIMATE"],
        "name_patterns": ["governance estimate", "estimate"],
        "measure": "estimate",
        "wide_suffix": "estimate",
    },
]

TARGET_WGI_MEASURES = [
    "estimate",
    "standard_error",
    "number_of_sources",
    "score",
    "score_lower",
    "score_upper",
]

print("Configured WGI dimensions:")
display(pd.DataFrame([
    {
        "dimension_code": code,
        "slug": meta["slug"],
        "name": meta["name"],
        "name_patterns": "; ".join(meta["name_patterns"]),
    }
    for code, meta in WGI_DIMENSIONS.items()
]))

print("Configured WGI measure rules:")
display(pd.DataFrame([
    {
        "measure": rule["measure"],
        "wide_suffix": rule["wide_suffix"],
        "code_suffixes": "; ".join(rule["code_suffixes"]),
        "name_patterns": "; ".join(rule["name_patterns"]),
    }
    for rule in WGI_MEASURE_RULES
]))

Configured WGI dimensions:


,dimension_code,slug,name,name_patterns
0,VA,va,Voice and Accountability,voice and accountability; voice accountability
1,PV,pv,Political Stability and Absence of Violence/Te...,political stability; absence of violence; terr...
2,GE,ge,Government Effectiveness,government effectiveness
3,RQ,rq,Regulatory Quality,regulatory quality
4,RL,rl,Rule of Law,rule of law
5,CC,cc,Control of Corruption,control of corruption; corruption


Configured WGI measure rules:


,measure,wide_suffix,code_suffixes,name_patterns
0,score_lower,score_lower,.SC_LB; .PER.RNK.LOWER; .PR.LOWER; .PERCENTILE...,lower bound of the 90% confidence interval for...
1,score_upper,score_upper,.SC_UB; .PER.RNK.UPPER; .PR.UPPER; .PERCENTILE...,upper bound of the 90% confidence interval for...
2,score,score,.SC; .PER.RNK; .PR; .PERCENTILE.RANK,governance score; score (0 100); score 0 100; ...
3,standard_error,std_error,.SE; .STD.ERR; .STANDARD.ERROR,standard error
4,number_of_sources,num_sources,.SR; .NO.SRC; .NOSRC; .NUMBER.SOURCES,number of sources; no. of sources; no of sources
5,estimate,estimate,.EST; .ESTIMATE,governance estimate; estimate


In [3]:
# ------------------------------------------------------
# PACKAGE EXTRACTION AND FILE INVENTORY
# ------------------------------------------------------

if not WGI_RAW_DIR.exists():
    WGI_RAW_DIR.mkdir(parents=True, exist_ok=True)

if len(list(WGI_RAW_DIR.rglob("*"))) == 0:
    if not WGI_ZIP_FILE.exists():
        raise FileNotFoundError(
            f"WGI raw folder is empty and WGI zip file was not found: {WGI_ZIP_FILE}"
        )

    if not zipfile.is_zipfile(WGI_ZIP_FILE):
        raise ValueError(f"WGI zip file exists but is not a valid zip: {WGI_ZIP_FILE}")

    with zipfile.ZipFile(WGI_ZIP_FILE, "r") as z:
        z.extractall(WGI_RAW_DIR)

csv_files = sorted(WGI_RAW_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in WGI raw folder: {WGI_RAW_DIR}")

inventory_rows = []

for path in csv_files:
    try:
        preview = pd.read_csv(path, nrows=5, encoding="utf-8-sig")
        rows_preview = len(preview)
        columns_preview = preview.columns.tolist()
    except Exception as e:
        rows_preview = np.nan
        columns_preview = [f"READ_ERROR: {e}"]

    inventory_rows.append({
        "file_name": path.name,
        "relative_path": str(path.relative_to(WGI_RAW_DIR)),
        "absolute_path": str(path),
        "size_bytes": path.stat().st_size,
        "preview_rows": rows_preview,
        "preview_columns": " | ".join(map(str, columns_preview[:30])),
    })

wgi_raw_file_inventory = pd.DataFrame(inventory_rows)

wgi_raw_file_inventory.to_csv(
    DIAGNOSTICS_DIR / "wgi_raw_file_inventory.csv",
    index=False,
)

display(wgi_raw_file_inventory)

,file_name,relative_path,absolute_path,size_bytes,preview_rows,preview_columns
0,WGICountry.csv,WGICountry.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,141210,5,Country Code | Short Name | Table Name | Long ...
1,WGICSV.csv,WGICSV.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,3412463,5,Country Name | Country Code | Indicator Name |...
2,WGISeries.csv,WGISeries.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,106435,5,Series Code | Topic | Indicator Name | Short d...


In [4]:
# ------------------------------------------------------
# CSV READING AND FILE DETECTION HELPERS
# ------------------------------------------------------

def read_csv_robust(path, nrows=None):
    encodings = ["utf-8-sig", "utf-8", "latin1"]

    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                encoding=encoding,
                nrows=nrows,
                low_memory=False,
            )
        except Exception as e:
            last_error = e

    raise last_error


def normalize_column_name(col):
    return str(col).strip().replace("\ufeff", "")


def clean_dataframe_columns(df):
    out = df.copy()
    out.columns = [normalize_column_name(c) for c in out.columns]
    return out


def find_first_existing_column(columns, candidates):
    normalized = {str(c).strip().lower(): c for c in columns}

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in normalized:
            return normalized[key]

    return None


def identify_year_columns(columns):
    year_cols = []

    for col in columns:
        col_str = str(col).strip()
        match = re.match(r"^(\d{4})(?:\s*\[YR\d{4}\])?$", col_str)

        if match:
            year = int(match.group(1))
            year_cols.append((col, year))

    return year_cols


def detect_wgi_candidate_file(path):
    try:
        preview = clean_dataframe_columns(read_csv_robust(path, nrows=500))
    except Exception as e:
        return {
            "file_name": path.name,
            "absolute_path": str(path),
            "readable": False,
            "score": -1,
            "error": str(e),
        }

    columns = preview.columns.tolist()

    country_col = find_first_existing_column(
        columns,
        ["Country Name", "Economy Name", "Country", "Economy"],
    )

    country_code_col = find_first_existing_column(
        columns,
        ["Country Code", "Economy Code", "REF_AREA"],
    )

    indicator_name_col = find_first_existing_column(
        columns,
        ["Indicator Name", "Series Name", "Governance dimension", "Indicator"],
    )

    indicator_code_col = find_first_existing_column(
        columns,
        ["Indicator Code", "Series Code", "Indicator_Code", "Series_Code"],
    )

    year_cols = identify_year_columns(columns)

    long_year_col = find_first_existing_column(
        columns,
        ["Year", "TIME_PERIOD", "Time"],
    )

    value_col = find_first_existing_column(
        columns,
        ["Value", "OBS_VALUE"],
    )

    is_wide_like = (
        country_code_col is not None
        and indicator_code_col is not None
        and len(year_cols) >= 5
    )

    is_long_like = (
        country_code_col is not None
        and indicator_code_col is not None
        and long_year_col is not None
        and value_col is not None
    )

    wgi_code_hits = 0
    wgi_name_hits = 0

    if indicator_code_col is not None and indicator_code_col in preview.columns:
        codes = preview[indicator_code_col].dropna().astype(str).str.upper()
        wgi_code_hits = int(codes.str.contains(r"^(VA|PV|GE|RQ|RL|CC)(\.|$)", regex=True).sum())

    if indicator_name_col is not None and indicator_name_col in preview.columns:
        names = preview[indicator_name_col].dropna().astype(str).str.lower()
        patterns = []
        for meta in WGI_DIMENSIONS.values():
            patterns.extend(meta["name_patterns"])
        combined_pattern = "|".join(re.escape(p) for p in patterns)
        wgi_name_hits = int(names.str.contains(combined_pattern, regex=True).sum())

    score = 0

    if is_wide_like:
        score += 100

    if is_long_like:
        score += 80

    score += len(year_cols)
    score += min(wgi_code_hits, 40)
    score += min(wgi_name_hits, 40)

    return {
        "file_name": path.name,
        "absolute_path": str(path),
        "readable": True,
        "score": score,
        "country_col": country_col,
        "country_code_col": country_code_col,
        "indicator_name_col": indicator_name_col,
        "indicator_code_col": indicator_code_col,
        "year_column_count": len(year_cols),
        "long_year_col": long_year_col,
        "value_col": value_col,
        "is_wide_like": is_wide_like,
        "is_long_like": is_long_like,
        "wgi_code_hits_preview": wgi_code_hits,
        "wgi_name_hits_preview": wgi_name_hits,
        "error": "",
    }


candidate_file_diagnostics = pd.DataFrame([
    detect_wgi_candidate_file(path)
    for path in csv_files
])

candidate_file_diagnostics = candidate_file_diagnostics.sort_values(
    ["score", "file_name"],
    ascending=[False, True],
).reset_index(drop=True)

candidate_file_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "wgi_candidate_data_file_diagnostics.csv",
    index=False,
)

display(candidate_file_diagnostics)

,file_name,absolute_path,readable,score,country_col,country_code_col,indicator_name_col,indicator_code_col,year_column_count,long_year_col,value_col,is_wide_like,is_long_like,wgi_code_hits_preview,wgi_name_hits_preview,error
0,WGICSV.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,True,166,Country Name,Country Code,Indicator Name,Indicator Code,26,None,None,True,False,0,500,
1,WGISeries.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,True,36,None,None,Indicator Name,Series Code,0,None,None,False,False,0,36,
2,WGICountry.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,True,0,None,Country Code,None,None,0,None,None,False,False,0,0,


In [5]:
# ------------------------------------------------------
# SELECT MAIN WGI DATA FILE
# ------------------------------------------------------

valid_candidates = candidate_file_diagnostics.query("readable == True and score > 0").copy()

if valid_candidates.empty:
    raise ValueError("No plausible WGI data file found. Review wgi_candidate_data_file_diagnostics.csv.")

main_file_row = valid_candidates.iloc[0].to_dict()
WGI_MAIN_DATA_FILE = Path(main_file_row["absolute_path"])

print("Selected WGI main data file:")
print(WGI_MAIN_DATA_FILE)

wgi_raw_data = clean_dataframe_columns(read_csv_robust(WGI_MAIN_DATA_FILE))

print("Raw WGI data shape:", wgi_raw_data.shape)
print("Columns:")
print(wgi_raw_data.columns.tolist()[:80])

display(wgi_raw_data.head())

Selected WGI main data file:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Raw\Raw_Files\WGI_2025_Raw\WGICSV.csv
Raw WGI data shape: (7776, 30)
Columns:
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1996', '1998', '2000', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


,Country Name,Country Code,Indicator Name,Indicator Code,1996,1998,2000,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Anguilla,AIA,Control of Corruption - Governance estimate (a...,GOV_WGI_CC.EST,NaN,NaN,NaN,NaN,NaN,0.7015,1.3271,1.3353,1.3294,1.3310,1.3309,1.3319,1.3295,1.3246,1.3250,1.1646,1.1685,1.1717,1.1694,1.1732,1.1785,0.5558,0.5532,1.1763,1.1733,1.1762
1,Anguilla,AIA,Control of Corruption - Governance score (0-100),GOV_WGI_CC.SC,NaN,NaN,NaN,NaN,NaN,62.0550,74.7178,74.8834,74.7648,74.7963,74.7952,74.8142,74.7665,74.6674,74.6750,71.4287,71.5074,71.5709,71.5255,71.6027,71.7090,59.1041,59.0521,71.6659,71.6045,71.6636
2,Anguilla,AIA,Control of Corruption - Lower bound of the 90%...,GOV_WGI_CC.SC_LB,NaN,NaN,NaN,NaN,NaN,47.6063,60.2690,60.4347,60.3160,60.3476,60.3465,60.3655,60.3178,60.2187,60.2262,56.9800,57.0587,57.1222,57.0768,57.1540,57.2603,44.6554,44.6034,57.2171,57.1557,57.2149
3,Anguilla,AIA,Control of Corruption - Number of sources,GOV_WGI_CC.SR,NaN,NaN,NaN,NaN,NaN,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
4,Anguilla,AIA,Control of Corruption - Standard error of the ...,GOV_WGI_CC.SE,NaN,NaN,NaN,NaN,NaN,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353


In [6]:
# ------------------------------------------------------
# WGI CODE, NAME, AND MEASURE PARSING
# ------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"[^a-z0-9\. /,()]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_year_from_column(col):
    col_str = str(col).strip()
    match = re.match(r"^(\d{4})(?:\s*\[YR\d{4}\])?$", col_str)

    if not match:
        return None

    return int(match.group(1))


def detect_dimension(indicator_code, indicator_name):
    code = "" if pd.isna(indicator_code) else str(indicator_code).strip().upper()
    name = normalize_text(indicator_name)

    # Handles both old codes like CC.EST and current codes like GOV_WGI_CC.EST.
    code_match = re.search(r"(?:^|[._])(VA|PV|GE|RQ|RL|CC)(?:[._]|$)", code)

    if code_match:
        return code_match.group(1)

    for dim_code, meta in WGI_DIMENSIONS.items():
        for pattern in meta["name_patterns"]:
            if pattern in name:
                return dim_code

    return None


def detect_measure(indicator_code, indicator_name):
    code = "" if pd.isna(indicator_code) else str(indicator_code).strip().upper()
    name = normalize_text(indicator_name)

    # Code suffix first: more reliable.
    # Lower/upper score bounds are intentionally listed before plain .SC.
    for rule in WGI_MEASURE_RULES:
        for suffix in rule["code_suffixes"]:
            if code.endswith(suffix):
                return {
                    "wgi_measure": rule["measure"],
                    "wide_suffix": rule["wide_suffix"],
                    "measure_parse_source": "indicator_code",
                }

    # Name patterns second.
    for rule in WGI_MEASURE_RULES:
        for pattern in rule["name_patterns"]:
            if pattern in name:
                return {
                    "wgi_measure": rule["measure"],
                    "wide_suffix": rule["wide_suffix"],
                    "measure_parse_source": "indicator_name",
                }

    # Conservative fallback:
    # If a WGI dimension is present but no measure is explicit, assume estimate only.
    # This prevents governance score rows from being mislabeled as estimates when "score" is visible.
    if any(
        pattern in name
        for meta in WGI_DIMENSIONS.values()
        for pattern in meta["name_patterns"]
    ):
        return {
            "wgi_measure": "estimate",
            "wide_suffix": "estimate",
            "measure_parse_source": "fallback_dimension_name_assume_estimate",
        }

    return None


def parse_wgi_indicator(indicator_code, indicator_name):
    dimension_code = detect_dimension(indicator_code, indicator_name)

    if dimension_code is None:
        return None

    measure_info = detect_measure(indicator_code, indicator_name)

    if measure_info is None:
        return None

    return {
        "indicator_code_clean": "" if pd.isna(indicator_code) else str(indicator_code).strip().upper(),
        "wgi_dimension_code": dimension_code,
        "wgi_dimension_slug": WGI_DIMENSIONS[dimension_code]["slug"],
        "wgi_dimension_name": WGI_DIMENSIONS[dimension_code]["name"],
        "wgi_measure": measure_info["wgi_measure"],
        "wide_suffix": measure_info["wide_suffix"],
        "dimension_parse_source": "indicator_code_or_name",
        "measure_parse_source": measure_info["measure_parse_source"],
    }


def extract_wgi_long_from_wide(df):
    columns = df.columns.tolist()

    country_name_col = find_first_existing_column(
        columns,
        ["Country Name", "Economy Name", "Country", "Economy"],
    )

    country_code_col = find_first_existing_column(
        columns,
        ["Country Code", "Economy Code", "REF_AREA"],
    )

    indicator_name_col = find_first_existing_column(
        columns,
        ["Indicator Name", "Series Name", "Governance dimension", "Indicator"],
    )

    indicator_code_col = find_first_existing_column(
        columns,
        ["Indicator Code", "Series Code", "Indicator_Code", "Series_Code"],
    )

    year_cols = identify_year_columns(columns)

    if country_code_col is None or indicator_code_col is None or len(year_cols) == 0:
        raise ValueError("Wide WGI file missing required country/indicator/year structure.")

    id_cols = [country_code_col, indicator_code_col]

    if country_name_col is not None:
        id_cols.append(country_name_col)

    if indicator_name_col is not None:
        id_cols.append(indicator_name_col)

    year_col_names = [col for col, year in year_cols]

    long = df[id_cols + year_col_names].copy()

    long = long.dropna(subset=[country_code_col, indicator_code_col]).copy()

    long = long.melt(
        id_vars=id_cols,
        value_vars=year_col_names,
        var_name="year_column",
        value_name="Value",
    )

    long["Year"] = long["year_column"].apply(parse_year_from_column)
    long["Value"] = pd.to_numeric(long["Value"], errors="coerce")

    rename_map = {
        country_code_col: "Country",
        indicator_code_col: "Indicator_Code",
    }

    if country_name_col is not None:
        rename_map[country_name_col] = "country_name"
    else:
        long["country_name"] = np.nan

    if indicator_name_col is not None:
        rename_map[indicator_name_col] = "Indicator_Name"
    else:
        long["Indicator_Name"] = np.nan

    long = long.rename(columns=rename_map)

    long = long.dropna(subset=["Country", "Indicator_Code", "Year"]).copy()

    return long


def extract_wgi_long_from_long(df):
    columns = df.columns.tolist()

    country_name_col = find_first_existing_column(
        columns,
        ["Country Name", "Economy Name", "Country", "Economy"],
    )

    country_code_col = find_first_existing_column(
        columns,
        ["Country Code", "Economy Code", "REF_AREA"],
    )

    indicator_name_col = find_first_existing_column(
        columns,
        ["Indicator Name", "Series Name", "Governance dimension", "Indicator"],
    )

    indicator_code_col = find_first_existing_column(
        columns,
        ["Indicator Code", "Series Code", "Indicator_Code", "Series_Code"],
    )

    year_col = find_first_existing_column(
        columns,
        ["Year", "TIME_PERIOD", "Time"],
    )

    value_col = find_first_existing_column(
        columns,
        ["Value", "OBS_VALUE"],
    )

    if country_code_col is None or indicator_code_col is None or year_col is None or value_col is None:
        raise ValueError("Long WGI file missing required country/indicator/year/value structure.")

    out = df.copy()

    rename_map = {
        country_code_col: "Country",
        indicator_code_col: "Indicator_Code",
        year_col: "Year",
        value_col: "Value",
    }

    if country_name_col is not None:
        rename_map[country_name_col] = "country_name"
    else:
        out["country_name"] = np.nan

    if indicator_name_col is not None:
        rename_map[indicator_name_col] = "Indicator_Name"
    else:
        out["Indicator_Name"] = np.nan

    out = out.rename(columns=rename_map)

    out["Year"] = pd.to_numeric(out["Year"], errors="coerce")
    out["Value"] = pd.to_numeric(out["Value"], errors="coerce")

    out = out.dropna(subset=["Country", "Indicator_Code", "Year"]).copy()
    out["Year"] = out["Year"].astype(int)

    return out[["Country", "country_name", "Indicator_Name", "Indicator_Code", "Year", "Value"]].copy()

In [7]:
# ------------------------------------------------------
# EXTRACT SELECTED WGI LONG DATA
# ------------------------------------------------------

selected_shape = "wide" if bool(main_file_row.get("is_wide_like")) else "long"

print("Detected selected data shape:", selected_shape)

if selected_shape == "wide":
    wgi_long_raw = extract_wgi_long_from_wide(wgi_raw_data)
else:
    wgi_long_raw = extract_wgi_long_from_long(wgi_raw_data)

wgi_long_raw["Country"] = wgi_long_raw["Country"].astype(str).str.strip().str.upper()
wgi_long_raw["country_name"] = wgi_long_raw["country_name"].astype(str).str.strip()
wgi_long_raw["Indicator_Code"] = wgi_long_raw["Indicator_Code"].astype(str).str.strip().str.upper()
wgi_long_raw["Indicator_Name"] = wgi_long_raw["Indicator_Name"].astype(str).str.strip()

unique_indicators = (
    wgi_long_raw[["Indicator_Code", "Indicator_Name"]]
    .drop_duplicates()
    .sort_values(["Indicator_Code", "Indicator_Name"])
    .reset_index(drop=True)
)

parsed_indicator_rows = []

for _, row in unique_indicators.iterrows():
    parsed = parse_wgi_indicator(
        indicator_code=row["Indicator_Code"],
        indicator_name=row["Indicator_Name"],
    )

    parsed_indicator_rows.append({
        "Indicator_Code": row["Indicator_Code"],
        "Indicator_Name": row["Indicator_Name"],
        **(parsed if parsed is not None else {}),
        "parsed_successfully": parsed is not None,
    })

wgi_indicator_parse_diagnostics = pd.DataFrame(parsed_indicator_rows)

wgi_indicator_parse_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "wgi_indicator_parse_diagnostics.csv",
    index=False,
)

unparsed_indicators = wgi_indicator_parse_diagnostics[
    ~wgi_indicator_parse_diagnostics["parsed_successfully"]
].copy()

unparsed_indicators.to_csv(
    DIAGNOSTICS_DIR / "wgi_unparsed_indicators.csv",
    index=False,
)

parsed_indicator_map = wgi_indicator_parse_diagnostics[
    wgi_indicator_parse_diagnostics["parsed_successfully"]
].copy()

merge_cols = [
    "Indicator_Code",
    "Indicator_Name",
    "indicator_code_clean",
    "wgi_dimension_code",
    "wgi_dimension_slug",
    "wgi_dimension_name",
    "wgi_measure",
    "wide_suffix",
    "dimension_parse_source",
    "measure_parse_source",
]

wgi_long = wgi_long_raw.merge(
    parsed_indicator_map[merge_cols],
    on=["Indicator_Code", "Indicator_Name"],
    how="inner",
)

wgi_long = wgi_long.dropna(
    subset=[
        "wgi_dimension_code",
        "wgi_measure",
        "Year",
    ]
).copy()

wgi_long = wgi_long[wgi_long["wgi_measure"].isin(TARGET_WGI_MEASURES)].copy()

wgi_long["Year"] = wgi_long["Year"].astype(int)

wgi_long = wgi_long[
    (wgi_long["Year"] >= START_YEAR)
    & (wgi_long["Year"] <= PROJECT_END_YEAR)
].copy()

wgi_long = wgi_long[
    [
        "Country",
        "country_name",
        "Year",
        "wgi_dimension_code",
        "wgi_dimension_slug",
        "wgi_dimension_name",
        "wgi_measure",
        "wide_suffix",
        "Indicator_Code",
        "Indicator_Name",
        "Value",
        "dimension_parse_source",
        "measure_parse_source",
    ]
].sort_values(["Country", "Year", "wgi_dimension_code", "wgi_measure"]).reset_index(drop=True)

if wgi_long.empty:
    raise ValueError(
        "No WGI rows were parsed. Review diagnostics: "
        "wgi_indicator_parse_diagnostics.csv and wgi_unparsed_indicators.csv"
    )

wgi_long.to_csv(
    PROCESSED_DIR / "wgi_long_selected_measures.csv",
    index=False,
)

wgi_long.to_csv(
    DIAGNOSTICS_DIR / "wgi_long_selected_measures.csv",
    index=False,
)

print("Unique WGI indicators found:")
display(wgi_indicator_parse_diagnostics)

print("Unparsed indicators:")
display(unparsed_indicators)

print("WGI long rows:", len(wgi_long))
print("Countries/economies:", wgi_long["Country"].nunique())
print("Years:", wgi_long["Year"].min(), "-", wgi_long["Year"].max())
print("Dimensions:", sorted(wgi_long["wgi_dimension_code"].dropna().unique()))
print("Measures:", sorted(wgi_long["wgi_measure"].dropna().unique()))

display(wgi_long.head(50))

Detected selected data shape: wide
Unique WGI indicators found:


,Indicator_Code,Indicator_Name,indicator_code_clean,wgi_dimension_code,wgi_dimension_slug,wgi_dimension_name,wgi_measure,wide_suffix,dimension_parse_source,measure_parse_source,parsed_successfully
0,GOV_WGI_CC.EST,Control of Corruption - Governance estimate (a...,GOV_WGI_CC.EST,CC,cc,Control of Corruption,estimate,estimate,indicator_code_or_name,indicator_code,True
1,GOV_WGI_CC.SC,Control of Corruption - Governance score (0-100),GOV_WGI_CC.SC,CC,cc,Control of Corruption,score,score,indicator_code_or_name,indicator_code,True
2,GOV_WGI_CC.SC_LB,Control of Corruption - Lower bound of the 90%...,GOV_WGI_CC.SC_LB,CC,cc,Control of Corruption,score_lower,score_lower,indicator_code_or_name,indicator_code,True
3,GOV_WGI_CC.SC_UB,Control of Corruption - Upper bound of the 90%...,GOV_WGI_CC.SC_UB,CC,cc,Control of Corruption,score_upper,score_upper,indicator_code_or_name,indicator_code,True
4,GOV_WGI_CC.SE,Control of Corruption - Standard error of the ...,GOV_WGI_CC.SE,CC,cc,Control of Corruption,standard_error,std_error,indicator_code_or_name,indicator_code,True
5,GOV_WGI_CC.SR,Control of Corruption - Number of sources,GOV_WGI_CC.SR,CC,cc,Control of Corruption,number_of_sources,num_sources,indicator_code_or_name,indicator_code,True
6,GOV_WGI_GE.EST,Government Effectiveness - Governance estimate...,GOV_WGI_GE.EST,GE,ge,Government Effectiveness,estimate,estimate,indicator_code_or_name,indicator_code,True
7,GOV_WGI_GE.SC,Government Effectiveness - Governance score (0...,GOV_WGI_GE.SC,GE,ge,Government Effectiveness,score,score,indicator_code_or_name,indicator_code,True
8,GOV_WGI_GE.SC_LB,Government Effectiveness - Lower bound of the ...,GOV_WGI_GE.SC_LB,GE,ge,Government Effectiveness,score_lower,score_lower,indicator_code_or_name,indicator_code,True
9,GOV_WGI_GE.SC_UB,Government Effectiveness - Upper bound of the ...,GOV_WGI_GE.SC_UB,GE,ge,Government Effectiveness,score_upper,score_upper,indicator_code_or_name,indicator_code,True


Unparsed indicators:


,Indicator_Code,Indicator_Name,indicator_code_clean,wgi_dimension_code,wgi_dimension_slug,wgi_dimension_name,wgi_measure,wide_suffix,dimension_parse_source,measure_parse_source,parsed_successfully


WGI long rows: 202176
Countries/economies: 216
Years: 1996 - 2024
Dimensions: ['CC', 'GE', 'PV', 'RL', 'RQ', 'VA']
Measures: ['estimate', 'number_of_sources', 'score', 'score_lower', 'score_upper', 'standard_error']


,Country,country_name,Year,wgi_dimension_code,wgi_dimension_slug,wgi_dimension_name,wgi_measure,wide_suffix,Indicator_Code,Indicator_Name,Value,dimension_parse_source,measure_parse_source
0,ABW,Aruba,1996,CC,cc,Control of Corruption,estimate,estimate,GOV_WGI_CC.EST,Control of Corruption - Governance estimate (a...,NaN,indicator_code_or_name,indicator_code
1,ABW,Aruba,1996,CC,cc,Control of Corruption,number_of_sources,num_sources,GOV_WGI_CC.SR,Control of Corruption - Number of sources,NaN,indicator_code_or_name,indicator_code
2,ABW,Aruba,1996,CC,cc,Control of Corruption,score,score,GOV_WGI_CC.SC,Control of Corruption - Governance score (0-100),NaN,indicator_code_or_name,indicator_code
3,ABW,Aruba,1996,CC,cc,Control of Corruption,score_lower,score_lower,GOV_WGI_CC.SC_LB,Control of Corruption - Lower bound of the 90%...,NaN,indicator_code_or_name,indicator_code
4,ABW,Aruba,1996,CC,cc,Control of Corruption,score_upper,score_upper,GOV_WGI_CC.SC_UB,Control of Corruption - Upper bound of the 90%...,NaN,indicator_code_or_name,indicator_code
5,ABW,Aruba,1996,CC,cc,Control of Corruption,standard_error,std_error,GOV_WGI_CC.SE,Control of Corruption - Standard error of the ...,NaN,indicator_code_or_name,indicator_code
6,ABW,Aruba,1996,GE,ge,Government Effectiveness,estimate,estimate,GOV_WGI_GE.EST,Government Effectiveness - Governance estimate...,NaN,indicator_code_or_name,indicator_code
7,ABW,Aruba,1996,GE,ge,Government Effectiveness,number_of_sources,num_sources,GOV_WGI_GE.SR,Government Effectiveness - Number of sources,NaN,indicator_code_or_name,indicator_code
8,ABW,Aruba,1996,GE,ge,Government Effectiveness,score,score,GOV_WGI_GE.SC,Government Effectiveness - Governance score (0...,NaN,indicator_code_or_name,indicator_code
9,ABW,Aruba,1996,GE,ge,Government Effectiveness,score_lower,score_lower,GOV_WGI_GE.SC_LB,Government Effectiveness - Lower bound of the ...,NaN,indicator_code_or_name,indicator_code


In [8]:
# ------------------------------------------------------
# VALIDATE REQUIRED WGI DIMENSIONS AND AVAILABLE MEASURES
# ------------------------------------------------------

dimension_measure_presence = (
    wgi_long
    .groupby(["wgi_dimension_code", "wgi_measure"])
    .agg(
        rows=("Value", "size"),
        countries=("Country", "nunique"),
        years=("Year", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        min_value=("Value", "min"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .sort_values(["wgi_dimension_code", "wgi_measure"])
)

dimension_measure_presence.to_csv(
    DIAGNOSTICS_DIR / "wgi_dimension_measure_presence.csv",
    index=False,
)

required_dimension_codes = set(WGI_DIMENSIONS.keys())
found_dimension_codes = set(wgi_long["wgi_dimension_code"].dropna().unique())

missing_dimensions = sorted(required_dimension_codes - found_dimension_codes)

if missing_dimensions:
    raise ValueError(f"Missing WGI dimensions in extracted data: {missing_dimensions}")

available_measures = sorted(wgi_long["wgi_measure"].dropna().unique())

found_score_dimensions = set(
    wgi_long.loc[wgi_long["wgi_measure"] == "score", "wgi_dimension_code"]
    .dropna()
    .unique()
)

found_estimate_dimensions = set(
    wgi_long.loc[wgi_long["wgi_measure"] == "estimate", "wgi_dimension_code"]
    .dropna()
    .unique()
)

has_all_score_dimensions = found_score_dimensions == required_dimension_codes
has_all_estimate_dimensions = found_estimate_dimensions == required_dimension_codes

if not has_all_score_dimensions and not has_all_estimate_dimensions:
    raise ValueError(
        "The package does not contain all six WGI dimensions for either 0-100 governance scores or estimates. "
        "Review wgi_dimension_measure_presence.csv."
    )

if has_all_score_dimensions:
    GOVERNANCE_COMPOSITE_INPUT = "score"
    GOVERNANCE_COMPOSITE_MEASURE = "score"
    GOVERNANCE_COMPOSITE_SUFFIX = "score"
    GOVERNANCE_IDEAL_RULE = "fixed_100_for_governance_score"
    print("Composite input selected: WGI governance score / 0-100 columns.")
else:
    GOVERNANCE_COMPOSITE_INPUT = "estimate"
    GOVERNANCE_COMPOSITE_MEASURE = "estimate"
    GOVERNANCE_COMPOSITE_SUFFIX = "estimate"
    GOVERNANCE_IDEAL_RULE = "observed_dimensionwise_max_for_estimate"
    print("Composite input selected: WGI estimate columns.")
    print("Reason: 0-100 governance score rows were not available for all six dimensions.")

measure_availability_summary = pd.DataFrame([{
    "available_measures": ",".join(available_measures),
    "has_all_score_dimensions": has_all_score_dimensions,
    "has_all_estimate_dimensions": has_all_estimate_dimensions,
    "selected_composite_input": GOVERNANCE_COMPOSITE_INPUT,
    "selected_composite_measure": GOVERNANCE_COMPOSITE_MEASURE,
    "selected_ideal_rule": GOVERNANCE_IDEAL_RULE,
}])

measure_availability_summary.to_csv(
    DIAGNOSTICS_DIR / "wgi_measure_availability_summary.csv",
    index=False,
)

display(dimension_measure_presence)
display(measure_availability_summary)

Composite input selected: WGI governance score / 0-100 columns.


,wgi_dimension_code,wgi_measure,rows,countries,years,min_year,max_year,min_value,max_value
0,CC,estimate,5616,216,26,1996,2024,-2.0647,2.4839
1,CC,number_of_sources,5616,216,26,1996,2024,1.0000,15.0000
2,CC,score,5616,216,26,1996,2024,6.0617,98.1320
3,CC,score_lower,5616,216,26,1996,2024,0.0000,91.5039
4,CC,score_upper,5616,216,26,1996,2024,13.4356,100.0000
5,CC,standard_error,5616,216,26,1996,2024,0.1478,0.9812
6,GE,estimate,5616,216,26,1996,2024,-2.4619,2.3452
7,GE,number_of_sources,5616,216,26,1996,2024,1.0000,14.0000
8,GE,score,5616,216,26,1996,2024,3.4711,96.9403
9,GE,score_lower,5616,216,26,1996,2024,0.0000,88.8806


,available_measures,has_all_score_dimensions,has_all_estimate_dimensions,selected_composite_input,selected_composite_measure,selected_ideal_rule
0,"estimate,number_of_sources,score,score_lower,s...",True,True,score,score,fixed_100_for_governance_score


In [9]:
# ------------------------------------------------------
# PIVOT TO WIDE COUNTRY-YEAR FORMAT
# ------------------------------------------------------

wgi_long_for_pivot = wgi_long.copy()

wgi_long_for_pivot["wide_variable"] = (
    "wgi_"
    + wgi_long_for_pivot["wgi_dimension_slug"].astype(str)
    + "_"
    + wgi_long_for_pivot["wide_suffix"].astype(str)
)

duplicate_wgi_wide_check = (
    wgi_long_for_pivot
    .groupby(["Country", "Year", "wide_variable"])
    .agg(
        row_count=("Value", "size"),
        unique_values=("Value", "nunique"),
        min_value=("Value", "min"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .query("row_count > 1")
)

duplicate_wgi_wide_check.to_csv(
    DIAGNOSTICS_DIR / "wgi_duplicate_country_year_variable_check.csv",
    index=False,
)

if len(duplicate_wgi_wide_check.query("unique_values > 1")) > 0:
    display(duplicate_wgi_wide_check)
    raise ValueError("Conflicting duplicate WGI country-year-variable rows found.")

country_names = (
    wgi_long_for_pivot
    .groupby("Country")
    .agg(country_name=("country_name", "first"))
    .reset_index()
)

wgi_wide = (
    wgi_long_for_pivot
    .pivot_table(
        index=["Country", "Year"],
        columns="wide_variable",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)

wgi_wide = wgi_wide.merge(country_names, on="Country", how="left")

front_cols = ["Country", "country_name", "Year"]
other_cols = sorted([col for col in wgi_wide.columns if col not in front_cols])

wgi_wide = wgi_wide[front_cols + other_cols].sort_values(["Country", "Year"]).reset_index(drop=True)

score_cols = [f"wgi_{meta['slug']}_score" for meta in WGI_DIMENSIONS.values()]
estimate_cols = [f"wgi_{meta['slug']}_estimate" for meta in WGI_DIMENSIONS.values()]

composite_input_cols = [
    f"wgi_{meta['slug']}_{GOVERNANCE_COMPOSITE_SUFFIX}"
    for meta in WGI_DIMENSIONS.values()
]

# Add missing optional columns as NaN so downstream exports are stable.
for col in score_cols + estimate_cols:
    if col not in wgi_wide.columns:
        wgi_wide[col] = np.nan

wgi_wide.to_csv(
    PROCESSED_DIR / "wgi_country_year_scores_wide.csv",
    index=False,
)

wgi_wide_project_window = wgi_wide[
    (wgi_wide["Year"] >= PROJECT_START_YEAR)
    & (wgi_wide["Year"] <= PROJECT_END_YEAR)
].copy()

wgi_wide_project_window.to_csv(
    PROCESSED_DIR / "wgi_country_year_scores_wide_2000_2024.csv",
    index=False,
)

print("WGI wide rows:", len(wgi_wide))
print("Countries/economies:", wgi_wide["Country"].nunique())
print("Years:", wgi_wide["Year"].min(), "-", wgi_wide["Year"].max())
print("Composite input columns:")
print(composite_input_cols)

display(wgi_wide.head())

WGI wide rows: 5481
Countries/economies: 216
Years: 1996 - 2024
Composite input columns:
['wgi_va_score', 'wgi_pv_score', 'wgi_ge_score', 'wgi_rq_score', 'wgi_rl_score', 'wgi_cc_score']


,Country,country_name,Year,wgi_cc_estimate,wgi_cc_num_sources,wgi_cc_score,wgi_cc_score_lower,wgi_cc_score_upper,wgi_cc_std_error,wgi_ge_estimate,wgi_ge_num_sources,wgi_ge_score,wgi_ge_score_lower,wgi_ge_score_upper,wgi_ge_std_error,wgi_pv_estimate,wgi_pv_num_sources,wgi_pv_score,wgi_pv_score_lower,wgi_pv_score_upper,wgi_pv_std_error,wgi_rl_estimate,wgi_rl_num_sources,wgi_rl_score,wgi_rl_score_lower,wgi_rl_score_upper,wgi_rl_std_error,wgi_rq_estimate,wgi_rq_num_sources,wgi_rq_score,wgi_rq_score_lower,wgi_rq_score_upper,wgi_rq_std_error,wgi_va_estimate,wgi_va_num_sources,wgi_va_score,wgi_va_score_lower,wgi_va_score_upper,wgi_va_std_error
0,ABW,Aruba,2004,0.9663,1.0000,67.4139,52.9652,81.8627,0.4353,0.8951,1.0000,68.7456,54.5883,82.9030,0.4440,0.6292,1.0000,76.6424,65.0068,88.2779,0.4153,0.6070,1.0000,67.4120,53.7213,81.1027,0.4850,0.7433,1.0000,66.5859,54.0434,79.1283,0.4849,0.4432,1.0000,64.6072,51.4350,77.7793,0.4589
1,ABW,Aruba,2005,1.3271,1.0000,74.7178,60.2690,89.1665,0.4353,1.2501,1.0000,75.6468,61.4894,89.8042,0.4440,1.3615,2.0000,89.1503,79.6752,98.6254,0.3382,0.7886,1.0000,70.5375,56.8468,84.2282,0.4850,1.1126,1.0000,72.4106,59.8681,84.9530,0.4849,1.0541,1.0000,75.2981,62.1259,88.4703,0.4589
2,ABW,Aruba,2006,1.3353,1.0000,74.8834,60.4347,89.3321,0.4353,1.2471,1.0000,75.5895,61.4321,89.7469,0.4440,1.3644,2.0000,89.2011,79.7260,98.6762,0.3382,0.7860,1.0000,70.4922,56.8015,84.1829,0.4850,1.1088,1.0000,72.3508,59.8084,84.8933,0.4849,1.0648,1.0000,75.4853,62.3131,88.6575,0.4589
3,ABW,Aruba,2007,1.3294,1.0000,74.7648,60.3160,89.2135,0.4353,1.2516,1.0000,75.6761,61.5188,89.8335,0.4440,1.3694,2.0000,89.2851,79.8100,98.7602,0.3382,0.7912,1.0000,70.5815,56.8908,84.2722,0.4850,1.1154,1.0000,72.4548,59.9124,84.9973,0.4849,1.0617,1.0000,75.4311,62.2589,88.6033,0.4589
4,ABW,Aruba,2008,1.3310,1.0000,74.7963,60.3476,89.2450,0.4353,1.2551,1.0000,75.7437,61.5863,89.9011,0.4440,1.3687,2.0000,89.2748,79.7997,98.7499,0.3382,0.7864,1.0000,70.4990,56.8083,84.1897,0.4850,1.1125,1.0000,72.4087,59.8662,84.9512,0.4849,1.0591,1.0000,75.3868,62.2146,88.5590,0.4589


In [10]:
# ------------------------------------------------------
# REQUESTED SIX-DIMENSION STRUCTURES
# ------------------------------------------------------

estimate_requested_cols = ["Country", "country_name", "Year"] + estimate_cols
score_requested_cols = ["Country", "country_name", "Year"] + score_cols
selected_requested_cols = ["Country", "country_name", "Year"] + composite_input_cols

wgi_country_year_estimates_requested_structure = wgi_wide[estimate_requested_cols].copy()
wgi_country_year_scores_requested_structure = wgi_wide[score_requested_cols].copy()
wgi_country_year_selected_governance_structure = wgi_wide[selected_requested_cols].copy()

wgi_country_year_estimates_requested_structure.to_csv(
    PROCESSED_DIR / "wgi_country_year_estimates_requested_structure.csv",
    index=False,
)

wgi_country_year_scores_requested_structure.to_csv(
    PROCESSED_DIR / "wgi_country_year_scores_requested_structure.csv",
    index=False,
)

wgi_country_year_selected_governance_structure.to_csv(
    PROCESSED_DIR / "wgi_country_year_selected_governance_structure.csv",
    index=False,
)

# Backward-compatible file name.
# If score rows exist, this contains scores. If not, it contains estimates and the column names reveal that.
wgi_country_year_requested_structure = wgi_country_year_selected_governance_structure.copy()

wgi_country_year_requested_structure.to_csv(
    PROCESSED_DIR / "wgi_country_year_requested_structure.csv",
    index=False,
)

wgi_country_year_requested_structure_2000_2024 = wgi_country_year_requested_structure[
    (wgi_country_year_requested_structure["Year"] >= PROJECT_START_YEAR)
    & (wgi_country_year_requested_structure["Year"] <= PROJECT_END_YEAR)
].copy()

wgi_country_year_requested_structure_2000_2024.to_csv(
    PROCESSED_DIR / "wgi_country_year_requested_structure_2000_2024.csv",
    index=False,
)

display(wgi_country_year_requested_structure.head())

,Country,country_name,Year,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score
0,ABW,Aruba,2004,64.6072,76.6424,68.7456,66.5859,67.4120,67.4139
1,ABW,Aruba,2005,75.2981,89.1503,75.6468,72.4106,70.5375,74.7178
2,ABW,Aruba,2006,75.4853,89.2011,75.5895,72.3508,70.4922,74.8834
3,ABW,Aruba,2007,75.4311,89.2851,75.6761,72.4548,70.5815,74.7648
4,ABW,Aruba,2008,75.3868,89.2748,75.7437,72.4087,70.4990,74.7963


In [11]:
# ------------------------------------------------------
# WGI COVERAGE DIAGNOSTICS
# ------------------------------------------------------

wgi_coverage_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "timestamp": RUN_TIMESTAMP,
    "main_data_file": str(WGI_MAIN_DATA_FILE),
    "long_rows": len(wgi_long),
    "wide_country_year_rows": len(wgi_wide),
    "countries": wgi_wide["Country"].nunique(),
    "min_year": wgi_wide["Year"].min(),
    "max_year": wgi_wide["Year"].max(),
    "available_measures": ",".join(available_measures),
    "has_all_score_dimensions": has_all_score_dimensions,
    "has_all_estimate_dimensions": has_all_estimate_dimensions,
    "selected_composite_input": GOVERNANCE_COMPOSITE_INPUT,
    "selected_composite_columns": ",".join(composite_input_cols),
}])

wgi_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "wgi_coverage_summary.csv",
    index=False,
)

wgi_coverage_by_country = (
    wgi_wide
    .assign(
        complete_score_dimensions=wgi_wide[score_cols].notna().sum(axis=1),
        complete_estimate_dimensions=wgi_wide[estimate_cols].notna().sum(axis=1),
        complete_selected_dimensions=wgi_wide[composite_input_cols].notna().sum(axis=1),
    )
    .groupby(["Country", "country_name"])
    .agg(
        years_available=("Year", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        mean_complete_score_dimensions=("complete_score_dimensions", "mean"),
        min_complete_score_dimensions=("complete_score_dimensions", "min"),
        mean_complete_estimate_dimensions=("complete_estimate_dimensions", "mean"),
        min_complete_estimate_dimensions=("complete_estimate_dimensions", "min"),
        mean_complete_selected_dimensions=("complete_selected_dimensions", "mean"),
        min_complete_selected_dimensions=("complete_selected_dimensions", "min"),
    )
    .reset_index()
    .sort_values(["years_available", "Country"], ascending=[False, True])
)

wgi_coverage_by_country.to_csv(
    DIAGNOSTICS_DIR / "wgi_coverage_by_country.csv",
    index=False,
)

wgi_coverage_by_year = (
    wgi_wide
    .assign(
        has_all_score_dimensions=wgi_wide[score_cols].notna().all(axis=1),
        has_all_estimate_dimensions=wgi_wide[estimate_cols].notna().all(axis=1),
        has_all_selected_dimensions=wgi_wide[composite_input_cols].notna().all(axis=1),
    )
    .groupby("Year")
    .agg(
        countries=("Country", "nunique"),
        countries_with_all_score_dimensions=("has_all_score_dimensions", "sum"),
        countries_with_all_estimate_dimensions=("has_all_estimate_dimensions", "sum"),
        countries_with_all_selected_dimensions=("has_all_selected_dimensions", "sum"),
    )
    .reset_index()
)

wgi_coverage_by_year.to_csv(
    DIAGNOSTICS_DIR / "wgi_coverage_by_year.csv",
    index=False,
)

wgi_coverage_by_dimension = (
    wgi_long
    .groupby(["wgi_dimension_code", "wgi_dimension_slug", "wgi_dimension_name", "wgi_measure"])
    .agg(
        rows=("Value", "size"),
        countries=("Country", "nunique"),
        years=("Year", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        min_value=("Value", "min"),
        median_value=("Value", "median"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .sort_values(["wgi_dimension_code", "wgi_measure"])
)

wgi_coverage_by_dimension.to_csv(
    DIAGNOSTICS_DIR / "wgi_coverage_by_dimension.csv",
    index=False,
)

wgi_missingness_by_country_year = wgi_wide[["Country", "country_name", "Year"]].copy()
wgi_missingness_by_country_year["missing_score_dimensions"] = wgi_wide[score_cols].isna().sum(axis=1)
wgi_missingness_by_country_year["missing_estimate_dimensions"] = wgi_wide[estimate_cols].isna().sum(axis=1)
wgi_missingness_by_country_year["missing_selected_dimensions"] = wgi_wide[composite_input_cols].isna().sum(axis=1)
wgi_missingness_by_country_year["has_all_score_dimensions"] = wgi_missingness_by_country_year["missing_score_dimensions"].eq(0)
wgi_missingness_by_country_year["has_all_estimate_dimensions"] = wgi_missingness_by_country_year["missing_estimate_dimensions"].eq(0)
wgi_missingness_by_country_year["has_all_selected_dimensions"] = wgi_missingness_by_country_year["missing_selected_dimensions"].eq(0)

wgi_missingness_by_country_year.to_csv(
    DIAGNOSTICS_DIR / "wgi_missingness_by_country_year.csv",
    index=False,
)

display(wgi_coverage_summary)
display(wgi_coverage_by_country.head(50))
display(wgi_coverage_by_year.head())

,run_id,timestamp,main_data_file,long_rows,wide_country_year_rows,countries,min_year,max_year,available_measures,has_all_score_dimensions,has_all_estimate_dimensions,selected_composite_input,selected_composite_columns
0,20260622_032632,2026-06-22 03:26:32,D:\Milano Bicocca\Course Materials\1st Year\02...,202176,5481,216,1996,2024,"estimate,number_of_sources,score,score_lower,s...",True,True,score,"wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_..."


,Country,country_name,years_available,min_year,max_year,mean_complete_score_dimensions,min_complete_score_dimensions,mean_complete_estimate_dimensions,min_complete_estimate_dimensions,mean_complete_selected_dimensions,min_complete_selected_dimensions
1,AFG,Afghanistan,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
2,AGO,Angola,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
4,ALB,Albania,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
5,AND,Andorra,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
7,ARE,United Arab Emirates,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
8,ARG,Argentina,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
9,ARM,Armenia,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
11,ATG,Antigua and Barbuda,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
12,AUS,Australia,26,1996,2024,6.0000,6,6.0000,6,6.0000,6
13,AUT,Austria,26,1996,2024,6.0000,6,6.0000,6,6.0000,6


,Year,countries,countries_with_all_score_dimensions,countries_with_all_estimate_dimensions,countries_with_all_selected_dimensions
0,1996,201,184,184,184
1,1998,202,184,184,184
2,2000,202,184,184,184
3,2002,202,186,186,186
4,2003,202,186,186,186


In [12]:
# ------------------------------------------------------
# VALUE RANGE SANITY CHECKS
# ------------------------------------------------------

range_rows = []

for col in score_cols:
    values = pd.to_numeric(wgi_wide[col], errors="coerce")

    range_rows.append({
        "column": col,
        "expected_min": 0,
        "expected_max": 100,
        "rows_non_missing": values.notna().sum(),
        "min_value": values.min(),
        "median_value": values.median(),
        "max_value": values.max(),
        "below_expected_min": int((values < 0).sum()),
        "above_expected_max": int((values > 100).sum()),
        "note": "WGI governance score / 0-100 if available in package.",
    })

for col in estimate_cols:
    values = pd.to_numeric(wgi_wide[col], errors="coerce")

    range_rows.append({
        "column": col,
        "expected_min": -3,
        "expected_max": 3,
        "rows_non_missing": values.notna().sum(),
        "min_value": values.min(),
        "median_value": values.median(),
        "max_value": values.max(),
        "below_expected_min": int((values < -3).sum()),
        "above_expected_max": int((values > 3).sum()),
        "note": "WGI governance estimate; usually approximately -2.5 to 2.5.",
    })

wgi_value_range_sanity_checks = pd.DataFrame(range_rows)

wgi_value_range_sanity_checks.to_csv(
    DIAGNOSTICS_DIR / "wgi_value_range_sanity_checks.csv",
    index=False,
)

display(wgi_value_range_sanity_checks)

,column,expected_min,expected_max,rows_non_missing,min_value,median_value,max_value,below_expected_min,above_expected_max,note
0,wgi_va_score,0,100,5444,15.5240,57.4430,91.5816,0,0,WGI governance score / 0-100 if available in p...
1,wgi_pv_score,0,100,5429,13.7632,67.2859,95.9538,0,0,WGI governance score / 0-100 if available in p...
2,wgi_ge_score,0,100,5340,3.4711,49.0619,96.9403,0,0,WGI governance score / 0-100 if available in p...
3,wgi_rq_score,0,100,5341,8.3902,52.9778,90.5035,0,0,WGI governance score / 0-100 if available in p...
4,wgi_rl_score,0,100,5476,13.7996,54.5879,93.2613,0,0,WGI governance score / 0-100 if available in p...
5,wgi_cc_score,0,100,5373,6.0617,44.0110,98.1320,0,0,WGI governance score / 0-100 if available in p...
6,wgi_va_estimate,-3,3,5444,-2.3611,0.0339,1.9844,0,0,WGI governance estimate; usually approximately...
7,wgi_pv_estimate,-3,3,5429,-3.0519,0.0815,1.7598,3,0,WGI governance estimate; usually approximately...
8,wgi_ge_estimate,-3,3,5340,-2.4619,-0.1172,2.3452,0,0,WGI governance estimate; usually approximately...
9,wgi_rq_estimate,-3,3,5341,-2.9466,-0.1195,2.2598,0,0,WGI governance estimate; usually approximately...


In [13]:
# ------------------------------------------------------
# SIX-DIMENSION CORRELATION MATRICES
# ------------------------------------------------------

selected_complete = wgi_wide.dropna(subset=composite_input_cols).copy()
score_complete = wgi_wide.dropna(subset=score_cols).copy()
estimate_complete = wgi_wide.dropna(subset=estimate_cols).copy()

selected_corr = selected_complete[composite_input_cols].corr()

selected_corr.to_csv(
    PROCESSED_DIR / "wgi_six_dimension_correlation_matrix.csv"
)

selected_corr.to_csv(
    DIAGNOSTICS_DIR / "wgi_six_dimension_correlation_matrix.csv"
)

if len(score_complete) > 0:
    score_corr = score_complete[score_cols].corr()
else:
    score_corr = pd.DataFrame(index=score_cols, columns=score_cols)

score_corr.to_csv(
    DIAGNOSTICS_DIR / "wgi_six_score_correlation_matrix.csv"
)

if len(estimate_complete) > 0:
    estimate_corr = estimate_complete[estimate_cols].corr()
else:
    estimate_corr = pd.DataFrame(index=estimate_cols, columns=estimate_cols)

estimate_corr.to_csv(
    DIAGNOSTICS_DIR / "wgi_six_estimate_correlation_matrix.csv"
)

print("Selected composite input:", GOVERNANCE_COMPOSITE_INPUT)
print("Complete rows for selected correlation:", len(selected_complete))
print("Complete rows for score correlation:", len(score_complete))
print("Complete rows for estimate correlation:", len(estimate_complete))

display(selected_corr)

Selected composite input: score
Complete rows for selected correlation: 5296
Complete rows for score correlation: 5296
Complete rows for estimate correlation: 5296


,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score
wgi_va_score,1.0000,0.7322,0.7924,0.8147,0.8877,0.8267
wgi_pv_score,0.7322,1.0000,0.7246,0.6910,0.8198,0.7726
wgi_ge_score,0.7924,0.7246,1.0000,0.9321,0.9099,0.9215
wgi_rq_score,0.8147,0.6910,0.9321,1.0000,0.9032,0.8853
wgi_rl_score,0.8877,0.8198,0.9099,0.9032,1.0000,0.9385
wgi_cc_score,0.8267,0.7726,0.9215,0.8853,0.9385,1.0000


In [14]:
# ------------------------------------------------------
# GOVERNANCE COMPOSITES
# ------------------------------------------------------
# These are derived project composites.
# They are not official WGI indicators.
#
# Composite input:
# - WGI governance score / 0-100 columns, if available for all six dimensions; otherwise
# - WGI estimate columns.
#
# Ideal point:
# - fixed 100 if using percentile-rank scores;
# - observed dimension-wise maximum if using estimates.

def calculate_euclidean_distance_to_ideal(x, ideal):
    diff = x - ideal
    return np.sqrt(np.sum(diff ** 2, axis=1))


def calculate_mahalanobis_distance_to_ideal(x, ideal, covariance_matrix):
    inv_cov = np.linalg.pinv(covariance_matrix)
    diff = x - ideal
    left = diff @ inv_cov
    distances_sq = np.sum(left * diff, axis=1)
    distances_sq = np.maximum(distances_sq, 0)
    return np.sqrt(distances_sq)


wgi_with_composite = wgi_wide.copy()

complete_mask = wgi_with_composite[composite_input_cols].notna().all(axis=1)
complete_values = wgi_with_composite.loc[complete_mask, composite_input_cols].astype(float).copy()

if complete_values.empty:
    raise ValueError("No complete six-dimension WGI rows available for composite construction.")

value_matrix = complete_values.to_numpy()

if GOVERNANCE_COMPOSITE_INPUT == "score":
    ideal_vector = np.repeat(100.0, len(composite_input_cols))
    max_possible_euclidean_distance = 100.0 * np.sqrt(len(composite_input_cols))
else:
    ideal_vector = complete_values.max(axis=0).to_numpy()
    # For estimate input, scale Euclidean closeness by the maximum observed distance to the observed ideal.
    # This avoids pretending estimates are naturally 0-100.
    raw_euclidean_to_ideal = calculate_euclidean_distance_to_ideal(value_matrix, ideal_vector)
    max_possible_euclidean_distance = float(np.nanmax(raw_euclidean_to_ideal))
    if max_possible_euclidean_distance == 0:
        max_possible_euclidean_distance = 1.0

euclidean_distance = calculate_euclidean_distance_to_ideal(value_matrix, ideal_vector)

euclidean_ideal_score = 100.0 * (
    1.0 - euclidean_distance / max_possible_euclidean_distance
)

covariance_matrix = np.cov(value_matrix, rowvar=False)

mahalanobis_distance = calculate_mahalanobis_distance_to_ideal(
    x=value_matrix,
    ideal=ideal_vector,
    covariance_matrix=covariance_matrix,
)

max_observed_mahalanobis_distance = np.nanmax(mahalanobis_distance)

if max_observed_mahalanobis_distance > 0:
    mahalanobis_ideal_score = 100.0 * (
        1.0 - mahalanobis_distance / max_observed_mahalanobis_distance
    )
else:
    mahalanobis_ideal_score = np.repeat(100.0, len(mahalanobis_distance))

wgi_with_composite["wgi_complete_six_selected_dimensions"] = complete_mask
wgi_with_composite["wgi_composite_input_type"] = GOVERNANCE_COMPOSITE_INPUT

wgi_with_composite["wgi_euclidean_distance_to_ideal_selected_space"] = np.nan
wgi_with_composite["wgi_euclidean_ideal_score"] = np.nan
wgi_with_composite["wgi_mahalanobis_distance_to_ideal_full_wgi"] = np.nan
wgi_with_composite["wgi_mahalanobis_ideal_score_full_wgi"] = np.nan

wgi_with_composite.loc[complete_mask, "wgi_euclidean_distance_to_ideal_selected_space"] = euclidean_distance
wgi_with_composite.loc[complete_mask, "wgi_euclidean_ideal_score"] = euclidean_ideal_score
wgi_with_composite.loc[complete_mask, "wgi_mahalanobis_distance_to_ideal_full_wgi"] = mahalanobis_distance
wgi_with_composite.loc[complete_mask, "wgi_mahalanobis_ideal_score_full_wgi"] = mahalanobis_ideal_score

wgi_with_composite.to_csv(
    PROCESSED_DIR / "wgi_country_year_scores_with_composite.csv",
    index=False,
)

wgi_with_composite_2000_2024 = wgi_with_composite[
    (wgi_with_composite["Year"] >= PROJECT_START_YEAR)
    & (wgi_with_composite["Year"] <= PROJECT_END_YEAR)
].copy()

wgi_with_composite_2000_2024.to_csv(
    PROCESSED_DIR / "wgi_country_year_scores_with_composite_2000_2024.csv",
    index=False,
)

composite_parameters = pd.DataFrame([
    {
        "parameter": "composite_input_type",
        "value": GOVERNANCE_COMPOSITE_INPUT,
    },
    {
        "parameter": "composite_measure",
        "value": GOVERNANCE_COMPOSITE_MEASURE,
    },
    {
        "parameter": "composite_input_columns",
        "value": ",".join(composite_input_cols),
    },
    {
        "parameter": "ideal_rule",
        "value": GOVERNANCE_IDEAL_RULE,
    },
    {
        "parameter": "ideal_vector",
        "value": ",".join([str(round(v, 8)) for v in ideal_vector]),
    },
    {
        "parameter": "complete_rows_used_for_covariance",
        "value": len(complete_values),
    },
    {
        "parameter": "max_possible_or_observed_euclidean_distance",
        "value": max_possible_euclidean_distance,
    },
    {
        "parameter": "max_observed_mahalanobis_distance",
        "value": max_observed_mahalanobis_distance,
    },
    {
        "parameter": "mahalanobis_covariance_sample",
        "value": "Full complete six-dimension WGI sample in extracted package for selected composite input.",
    },
    {
        "parameter": "methodological_note",
        "value": "Mahalanobis governance score is a project-derived composite, not an official WGI indicator.",
    },
])

composite_parameters.to_csv(
    PROCESSED_DIR / "wgi_mahalanobis_composite_parameters.csv",
    index=False,
)

composite_parameters.to_csv(
    DIAGNOSTICS_DIR / "wgi_mahalanobis_composite_parameters.csv",
    index=False,
)

covariance_df = pd.DataFrame(
    covariance_matrix,
    index=composite_input_cols,
    columns=composite_input_cols,
)

covariance_df.to_csv(
    DIAGNOSTICS_DIR / "wgi_covariance_matrix_used_for_mahalanobis.csv"
)

display(wgi_with_composite.head())
display(composite_parameters)

,Country,country_name,Year,wgi_cc_estimate,wgi_cc_num_sources,wgi_cc_score,wgi_cc_score_lower,wgi_cc_score_upper,wgi_cc_std_error,wgi_ge_estimate,wgi_ge_num_sources,wgi_ge_score,wgi_ge_score_lower,wgi_ge_score_upper,wgi_ge_std_error,wgi_pv_estimate,wgi_pv_num_sources,wgi_pv_score,wgi_pv_score_lower,wgi_pv_score_upper,wgi_pv_std_error,wgi_rl_estimate,wgi_rl_num_sources,wgi_rl_score,wgi_rl_score_lower,wgi_rl_score_upper,wgi_rl_std_error,wgi_rq_estimate,wgi_rq_num_sources,wgi_rq_score,wgi_rq_score_lower,wgi_rq_score_upper,wgi_rq_std_error,wgi_va_estimate,wgi_va_num_sources,wgi_va_score,wgi_va_score_lower,wgi_va_score_upper,wgi_va_std_error,wgi_complete_six_selected_dimensions,wgi_composite_input_type,wgi_euclidean_distance_to_ideal_selected_space,wgi_euclidean_ideal_score,wgi_mahalanobis_distance_to_ideal_full_wgi,wgi_mahalanobis_ideal_score_full_wgi
0,ABW,Aruba,2004,0.9663,1.0000,67.4139,52.9652,81.8627,0.4353,0.8951,1.0000,68.7456,54.5883,82.9030,0.4440,0.6292,1.0000,76.6424,65.0068,88.2779,0.4153,0.6070,1.0000,67.4120,53.7213,81.1027,0.4850,0.7433,1.0000,66.5859,54.0434,79.1283,0.4849,0.4432,1.0000,64.6072,51.4350,77.7793,0.4589,True,score,77.5590,68.3367,2.4697,67.1844
1,ABW,Aruba,2005,1.3271,1.0000,74.7178,60.2690,89.1665,0.4353,1.2501,1.0000,75.6468,61.4894,89.8042,0.4440,1.3615,2.0000,89.1503,79.6752,98.6254,0.3382,0.7886,1.0000,70.5375,56.8468,84.2282,0.4850,1.1126,1.0000,72.4106,59.8681,84.9530,0.4849,1.0541,1.0000,75.2981,62.1259,88.4703,0.4589,True,score,59.9115,75.5412,2.6330,65.0153
2,ABW,Aruba,2006,1.3353,1.0000,74.8834,60.4347,89.3321,0.4353,1.2471,1.0000,75.5895,61.4321,89.7469,0.4440,1.3644,2.0000,89.2011,79.7260,98.6762,0.3382,0.7860,1.0000,70.4922,56.8015,84.1829,0.4850,1.1088,1.0000,72.3508,59.8084,84.8933,0.4849,1.0648,1.0000,75.4853,62.3131,88.6575,0.4589,True,score,59.8288,75.5750,2.6533,64.7456
3,ABW,Aruba,2007,1.3294,1.0000,74.7648,60.3160,89.2135,0.4353,1.2516,1.0000,75.6761,61.5188,89.8335,0.4440,1.3694,2.0000,89.2851,79.8100,98.7602,0.3382,0.7912,1.0000,70.5815,56.8908,84.2722,0.4850,1.1154,1.0000,72.4548,59.9124,84.9973,0.4849,1.0617,1.0000,75.4311,62.2589,88.6033,0.4589,True,score,59.7586,75.6036,2.6362,64.9726
4,ABW,Aruba,2008,1.3310,1.0000,74.7963,60.3476,89.2450,0.4353,1.2551,1.0000,75.7437,61.5863,89.9011,0.4440,1.3687,2.0000,89.2748,79.7997,98.7499,0.3382,0.7864,1.0000,70.4990,56.8083,84.1897,0.4850,1.1125,1.0000,72.4087,59.8662,84.9512,0.4849,1.0591,1.0000,75.3868,62.2146,88.5590,0.4589,True,score,59.7999,75.5868,2.6553,64.7188


,parameter,value
0,composite_input_type,score
1,composite_measure,score
2,composite_input_columns,"wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_..."
3,ideal_rule,fixed_100_for_governance_score
4,ideal_vector,"100.0,100.0,100.0,100.0,100.0,100.0"
5,complete_rows_used_for_covariance,5296
6,max_possible_or_observed_euclidean_distance,244.9490
7,max_observed_mahalanobis_distance,7.5261
8,mahalanobis_covariance_sample,Full complete six-dimension WGI sample in extr...
9,methodological_note,Mahalanobis governance score is a project-deri...


In [15]:
# ------------------------------------------------------
# FINAL PROJECT-READY WGI OUTPUT
# ------------------------------------------------------
# This is the compact WGI file most useful for later master dataset construction.

final_wgi_cols = (
    ["Country", "country_name", "Year"]
    + score_cols
    + estimate_cols
    + [
        "wgi_complete_six_selected_dimensions",
        "wgi_composite_input_type",
        "wgi_euclidean_distance_to_ideal_selected_space",
        "wgi_euclidean_ideal_score",
        "wgi_mahalanobis_distance_to_ideal_full_wgi",
        "wgi_mahalanobis_ideal_score_full_wgi",
    ]
)

wgi_final_project_ready = wgi_with_composite[final_wgi_cols].copy()

wgi_final_project_ready.to_csv(
    PROCESSED_DIR / "wgi_final_project_ready_country_year.csv",
    index=False,
)

wgi_final_project_ready_2000_2024 = wgi_final_project_ready[
    (wgi_final_project_ready["Year"] >= PROJECT_START_YEAR)
    & (wgi_final_project_ready["Year"] <= PROJECT_END_YEAR)
].copy()

wgi_final_project_ready_2000_2024.to_csv(
    PROCESSED_DIR / "wgi_final_project_ready_country_year_2000_2024.csv",
    index=False,
)

print("Final WGI project-ready rows:", len(wgi_final_project_ready))
print("Project-window rows:", len(wgi_final_project_ready_2000_2024))
print("Countries/economies:", wgi_final_project_ready["Country"].nunique())
print("Selected composite input:", GOVERNANCE_COMPOSITE_INPUT)

display(wgi_final_project_ready.head())

Final WGI project-ready rows: 5481
Project-window rows: 5078
Countries/economies: 216
Selected composite input: score


,Country,country_name,Year,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_va_estimate,wgi_pv_estimate,wgi_ge_estimate,wgi_rq_estimate,wgi_rl_estimate,wgi_cc_estimate,wgi_complete_six_selected_dimensions,wgi_composite_input_type,wgi_euclidean_distance_to_ideal_selected_space,wgi_euclidean_ideal_score,wgi_mahalanobis_distance_to_ideal_full_wgi,wgi_mahalanobis_ideal_score_full_wgi
0,ABW,Aruba,2004,64.6072,76.6424,68.7456,66.5859,67.4120,67.4139,0.4432,0.6292,0.8951,0.7433,0.6070,0.9663,True,score,77.5590,68.3367,2.4697,67.1844
1,ABW,Aruba,2005,75.2981,89.1503,75.6468,72.4106,70.5375,74.7178,1.0541,1.3615,1.2501,1.1126,0.7886,1.3271,True,score,59.9115,75.5412,2.6330,65.0153
2,ABW,Aruba,2006,75.4853,89.2011,75.5895,72.3508,70.4922,74.8834,1.0648,1.3644,1.2471,1.1088,0.7860,1.3353,True,score,59.8288,75.5750,2.6533,64.7456
3,ABW,Aruba,2007,75.4311,89.2851,75.6761,72.4548,70.5815,74.7648,1.0617,1.3694,1.2516,1.1154,0.7912,1.3294,True,score,59.7586,75.6036,2.6362,64.9726
4,ABW,Aruba,2008,75.3868,89.2748,75.7437,72.4087,70.4990,74.7963,1.0591,1.3687,1.2551,1.1125,0.7864,1.3310,True,score,59.7999,75.5868,2.6553,64.7188


In [16]:
# ------------------------------------------------------
# DATA DICTIONARY AND METHODOLOGICAL NOTES
# ------------------------------------------------------

data_dictionary_rows = []

for dim_code, meta in WGI_DIMENSIONS.items():
    slug = meta["slug"]
    name = meta["name"]

    data_dictionary_rows.append({
        "column": f"wgi_{slug}_score",
        "description": f"WGI governance score / 0-100 for {name}, if available in the CSV package. Higher = better.",
        "official_wgi_indicator": True,
        "derived_project_variable": False,
    })

    data_dictionary_rows.append({
        "column": f"wgi_{slug}_estimate",
        "description": f"WGI governance estimate for {name}. Higher = better.",
        "official_wgi_indicator": True,
        "derived_project_variable": False,
    })

data_dictionary_rows.extend([
    {
        "column": "wgi_euclidean_ideal_score",
        "description": "Project-derived Euclidean closeness-to-ideal governance score using the selected six WGI governance dimensions. Higher = closer to ideal.",
        "official_wgi_indicator": False,
        "derived_project_variable": True,
    },
    {
        "column": "wgi_mahalanobis_ideal_score_full_wgi",
        "description": "Project-derived Mahalanobis closeness-to-ideal governance score using the selected six WGI governance dimensions and covariance estimated from the full complete WGI sample. Higher = closer to ideal.",
        "official_wgi_indicator": False,
        "derived_project_variable": True,
    },
    {
        "column": "wgi_complete_six_selected_dimensions",
        "description": "Boolean flag showing whether all six selected WGI dimensions are present for the country-year.",
        "official_wgi_indicator": False,
        "derived_project_variable": True,
    },
    {
        "column": "wgi_composite_input_type",
        "description": "Indicates whether the governance composite used percentile-rank score columns or estimate columns.",
        "official_wgi_indicator": False,
        "derived_project_variable": True,
    },
])

wgi_data_dictionary = pd.DataFrame(data_dictionary_rows)

wgi_data_dictionary.to_csv(
    PROCESSED_DIR / "wgi_data_dictionary.csv",
    index=False,
)

methodological_notes = pd.DataFrame([
    {
        "topic": "WGI source format",
        "note": "This notebook uses the World Bank WGI CSV/DataBank bulk package, not the previous six-sheet Excel workbook.",
    },
    {
        "topic": "Measure availability",
        "note": f"The selected WGI package available measures were: {', '.join(available_measures)}.",
    },
    {
        "topic": "Composite input",
        "note": f"The governance composite input selected by the notebook was: {GOVERNANCE_COMPOSITE_INPUT}.",
    },
    {
        "topic": "Official vs derived variables",
        "note": "The six WGI dimensions are official WGI variables. The Euclidean and Mahalanobis governance scores are project-derived composites.",
    },
    {
        "topic": "Composite use",
        "note": "The derived governance composite may be used as one candidate structural governance-capacity variable, but should not be confused with a final Economic Resilience Index.",
    },
    {
        "topic": "Redundancy risk",
        "note": "Do not use all six WGI dimensions as separate POSet ordering variables unless explicitly doing a sensitivity analysis, because they are expected to be highly correlated.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "wgi_methodological_notes.csv",
    index=False,
)

display(wgi_data_dictionary)
display(methodological_notes)

,column,description,official_wgi_indicator,derived_project_variable
0,wgi_va_score,WGI governance score / 0-100 for Voice and Acc...,True,False
1,wgi_va_estimate,WGI governance estimate for Voice and Accounta...,True,False
2,wgi_pv_score,WGI governance score / 0-100 for Political Sta...,True,False
3,wgi_pv_estimate,WGI governance estimate for Political Stabilit...,True,False
4,wgi_ge_score,WGI governance score / 0-100 for Government Ef...,True,False
5,wgi_ge_estimate,WGI governance estimate for Government Effecti...,True,False
6,wgi_rq_score,WGI governance score / 0-100 for Regulatory Qu...,True,False
7,wgi_rq_estimate,WGI governance estimate for Regulatory Quality...,True,False
8,wgi_rl_score,"WGI governance score / 0-100 for Rule of Law, ...",True,False
9,wgi_rl_estimate,WGI governance estimate for Rule of Law. Highe...,True,False


,topic,note
0,WGI source format,This notebook uses the World Bank WGI CSV/Data...
1,Measure availability,The selected WGI package available measures we...
2,Composite input,The governance composite input selected by the...
3,Official vs derived variables,The six WGI dimensions are official WGI variab...
4,Composite use,The derived governance composite may be used a...
5,Redundancy risk,Do not use all six WGI dimensions as separate ...


In [17]:
# ------------------------------------------------------
# CREATE WGI AUDIT WORKBOOK
# ------------------------------------------------------

WGI_AUDIT_XLSX = AUDIT_DIR / "03_WGI_Governance_Compilation_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_raw_inventory",
        "description": "Inventory of CSV files found in the extracted WGI package.",
        "path": DIAGNOSTICS_DIR / "wgi_raw_file_inventory.csv",
    },
    {
        "sheet_name": "02_candidate_files",
        "description": "Diagnostics used to select the main WGI data file.",
        "path": DIAGNOSTICS_DIR / "wgi_candidate_data_file_diagnostics.csv",
    },
    {
        "sheet_name": "03_indicator_parse",
        "description": "Indicator parsing diagnostics.",
        "path": DIAGNOSTICS_DIR / "wgi_indicator_parse_diagnostics.csv",
    },
    {
        "sheet_name": "04_measure_summary",
        "description": "Measure availability and selected composite input.",
        "path": DIAGNOSTICS_DIR / "wgi_measure_availability_summary.csv",
    },
    {
        "sheet_name": "05_coverage_summary",
        "description": "Overall WGI coverage summary.",
        "path": DIAGNOSTICS_DIR / "wgi_coverage_summary.csv",
    },
    {
        "sheet_name": "06_dimension_presence",
        "description": "WGI dimension-measure presence table.",
        "path": DIAGNOSTICS_DIR / "wgi_dimension_measure_presence.csv",
    },
    {
        "sheet_name": "07_coverage_country",
        "description": "WGI coverage by country/economy.",
        "path": DIAGNOSTICS_DIR / "wgi_coverage_by_country.csv",
    },
    {
        "sheet_name": "08_coverage_year",
        "description": "WGI coverage by year.",
        "path": DIAGNOSTICS_DIR / "wgi_coverage_by_year.csv",
    },
    {
        "sheet_name": "09_coverage_dimension",
        "description": "WGI coverage by dimension and measure.",
        "path": DIAGNOSTICS_DIR / "wgi_coverage_by_dimension.csv",
    },
    {
        "sheet_name": "10_missing_country_year",
        "description": "Missingness by country-year.",
        "path": DIAGNOSTICS_DIR / "wgi_missingness_by_country_year.csv",
    },
    {
        "sheet_name": "11_value_ranges",
        "description": "WGI value range sanity checks.",
        "path": DIAGNOSTICS_DIR / "wgi_value_range_sanity_checks.csv",
    },
    {
        "sheet_name": "12_selected_corr",
        "description": "Correlation matrix for selected six WGI dimensions.",
        "path": DIAGNOSTICS_DIR / "wgi_six_dimension_correlation_matrix.csv",
    },
    {
        "sheet_name": "13_composite_params",
        "description": "Parameters used for Euclidean and Mahalanobis governance composites.",
        "path": DIAGNOSTICS_DIR / "wgi_mahalanobis_composite_parameters.csv",
    },
    {
        "sheet_name": "14_dictionary",
        "description": "WGI output data dictionary.",
        "path": PROCESSED_DIR / "wgi_data_dictionary.csv",
    },
    {
        "sheet_name": "15_method_notes",
        "description": "Methodological notes for decision log/report.",
        "path": DIAGNOSTICS_DIR / "wgi_methodological_notes.csv",
    },
]


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))


used_sheets = set()

with pd.ExcelWriter(WGI_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "03 WGI Governance Compilation Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for WGI CSV package compilation and composite construction.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("WGI audit workbook created:")
print(WGI_AUDIT_XLSX.resolve())

WGI audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\03_WGI_Governance_Compilation_Audit.xlsx


In [18]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("03 WGI GOVERNANCE COMPILATION COMPLETE")
print("=" * 80)

print("\nWGI main data file:")
print(WGI_MAIN_DATA_FILE)

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(WGI_AUDIT_XLSX.resolve())

print("\nMain processed outputs:")
main_outputs = [
    "wgi_long_selected_measures.csv",
    "wgi_country_year_scores_wide.csv",
    "wgi_country_year_scores_wide_2000_2024.csv",
    "wgi_country_year_estimates_requested_structure.csv",
    "wgi_country_year_scores_requested_structure.csv",
    "wgi_country_year_selected_governance_structure.csv",
    "wgi_country_year_requested_structure.csv",
    "wgi_country_year_requested_structure_2000_2024.csv",
    "wgi_country_year_scores_with_composite.csv",
    "wgi_country_year_scores_with_composite_2000_2024.csv",
    "wgi_final_project_ready_country_year.csv",
    "wgi_final_project_ready_country_year_2000_2024.csv",
    "wgi_six_dimension_correlation_matrix.csv",
    "wgi_mahalanobis_composite_parameters.csv",
    "wgi_data_dictionary.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nSummary:")
print("Rows in final WGI project-ready file:", len(wgi_final_project_ready))
print("Countries/economies:", wgi_final_project_ready["Country"].nunique())
print("Years:", wgi_final_project_ready["Year"].min(), "-", wgi_final_project_ready["Year"].max())
print("Selected composite input:", GOVERNANCE_COMPOSITE_INPUT)
print("Complete six selected-dimension rows:", int(wgi_final_project_ready["wgi_complete_six_selected_dimensions"].sum()))

print("\nImportant notes:")
print("1. This notebook used the WGI CSV/DataBank package, not the old six-sheet Excel workbook.")
print("2. If 0-100 governance scores were unavailable, WGI estimates were used for the governance composite.")
print("3. wgi_mahalanobis_ideal_score_full_wgi is project-derived, not official WGI.")
print("4. WGI has not yet been merged into the master dataset.")
print("5. Do not use all six WGI dimensions as separate POSet variables unless running a sensitivity analysis.")

print("\nNext notebook:")
print("04_Volatility_Features.ipynb")

03 WGI GOVERNANCE COMPILATION COMPLETE

WGI main data file:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Raw\Raw_Files\WGI_2025_Raw\WGICSV.csv

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\WGI_Governance

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\03_WGI_Governance_Compilation

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\03_WGI_Governance_Compilation_Audit.xlsx

Main processed outputs:
- FOUND: wgi_long_selected_measures.csv
- FOUND: wgi_country_year_scores_wide.csv
- FOUND: wgi_country_year_scores_wide_2000_2024.csv
- FOUND: wgi_country_year_estimates_requested_structure.csv
- FOUND: wgi_country_year_scores_request